# Gate Recommendation Model Endpoint Test

**Model:** Scoring optimization (availability + terminal matching + proximity + delay penalty)  
**Endpoint:** `GET /api/predictions/gates/{icao24}`  
**Purpose:** Recommend optimal gate assignments for incoming flights

## Model Overview

The gate recommendation model scores each available gate using a weighted formula:

| Factor | Weight | Logic |
|--------|--------|-------|
| Availability | 50% | Available = 0.5, Delayed = 0.2, Occupied/Maintenance = 0.0 (excluded) |
| Terminal match | 25% | International flight + Terminal B = 0.25, domestic + Terminal A = 0.25, mismatch = 0.1 |
| Runway proximity | 15% | Lower gate number = closer to runway = higher score |
| Delay penalty | 10% | Flight delay > 30 min = -0.1 |

## Gate Configuration

- **Terminal A** (Domestic): Gates A1-A5
- **Terminal B** (International): Gates B1-B5

## Gate Status Types

| Status | Description |
|--------|-------------|
| `available` | Gate is free for assignment |
| `occupied` | Gate is currently in use |
| `delayed` | Gate available but previous flight was delayed |
| `maintenance` | Gate under maintenance, not usable |

## International Flight Detection

Domestic airline prefixes: AAL, UAL, DAL, SWA, JBU, NKS, ASA, FFT, SKW  
Any other callsign prefix is classified as international.

In [ ]:
# Setup & Authentication
import json
import time

import httpx
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# --- Configuration ---
APP_BASE_URL = "https://airport-digital-twin-dev-7474645572615955.aws.databricksapps.com"

# --- Authentication ---
def get_token():
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        return w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    except Exception:
        pass
    try:
        import subprocess
        result = subprocess.run(
            ["databricks", "auth", "token", "--profile", "FEVM_SERVERLESS_STABLE"],
            capture_output=True, text=True, timeout=15,
        )
        if result.returncode == 0:
            return json.loads(result.stdout)["access_token"]
    except Exception:
        pass
    raise RuntimeError("No Databricks auth token available")

TOKEN = get_token()
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"Authenticated (token: {len(TOKEN)} chars)")
print(f"App: {APP_BASE_URL}")

## Endpoint Parameters

### `GET /api/predictions/gates/{icao24}`

**Path parameters:**

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `icao24` | string | Yes | ICAO24 address of the flight |

**Query parameters:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `top_k` | int | 3 | Number of gate recommendations (1-10) |

**Response schema: `List[GateRecommendationResponse]`**

```json
[
  {
    "gate_id": "A1",
    "score": 0.85,
    "reasons": [
      "Gate is currently available",
      "Domestic terminal matches flight type",
      "Close to runway for quick turnaround"
    ],
    "taxi_time": 5
  }
]
```

| Field | Type | Description |
|-------|------|-------------|
| `gate_id` | string | Recommended gate ID (e.g., A1, B3) |
| `score` | float | Recommendation score (0.0 - 1.0, higher is better) |
| `reasons` | list[str] | Human-readable explanations for the recommendation |
| `taxi_time` | int | Estimated taxi time from runway to gate (minutes) |

In [ ]:
# List Active Flights & Select One

# Fetch current flights to pick an icao24
flights_url = f"{APP_BASE_URL}/api/flights"
print(f"GET {flights_url}")

resp = httpx.get(flights_url, headers=HEADERS, timeout=30)
print(f"Status: {resp.status_code}")

flights = []
if resp.status_code == 200:
    data = resp.json()
    flights = data.get("flights", data) if isinstance(data, dict) else data
    print(f"Active flights: {len(flights)}")

    # Show first 10 flights
    header = "| # | ICAO24 | Callsign | On Ground | Altitude |"
    sep = "|---|--------|----------|-----------|----------|"
    rows = [header, sep]
    for i, f in enumerate(flights[:10]):
        rows.append(
            f"| {i+1} | `{f.get('icao24', '?')}` | {f.get('callsign', '?')} "
            f"| {f.get('on_ground', '?')} | {f.get('baro_altitude', '?')} |"
        )
    display(HTML("<pre>" + "\n".join(rows) + "</pre>"))

# --- Select a flight ---
if flights:
    # Pick the first arriving (not on ground) flight, or fall back to first
    arriving = [f for f in flights if not f.get("on_ground", True)]
    selected = arriving[0] if arriving else flights[0]
    SELECTED_ICAO24 = selected.get("icao24", "unknown")
    print(f"\nSelected flight: {SELECTED_ICAO24} ({selected.get('callsign', '?')})")
else:
    SELECTED_ICAO24 = "a12345"  # fallback
    print(f"No flights found, using placeholder: {SELECTED_ICAO24}")

In [ ]:
# Call Gate Recommendation Endpoint

TOP_K = 5
url = f"{APP_BASE_URL}/api/predictions/gates/{SELECTED_ICAO24}?top_k={TOP_K}"
print(f"GET {url}")

t0 = time.monotonic()
resp = httpx.get(url, headers=HEADERS, timeout=30)
elapsed_ms = int((time.monotonic() - t0) * 1000)

print(f"Status: {resp.status_code} ({elapsed_ms}ms)")

recommendations = []
if resp.status_code == 200:
    recommendations = resp.json()
    print(f"Recommendations returned: {len(recommendations)}")

    # Show as table
    header = "| Rank | Gate | Score | Taxi Time | Reasons |"
    sep = "|------|------|-------|-----------|----------|"
    rows = [header, sep]
    for i, rec in enumerate(recommendations):
        reasons = "; ".join(rec.get("reasons", []))
        rows.append(
            f"| {i+1} | **{rec['gate_id']}** | {rec['score']:.2f} "
            f"| {rec['taxi_time']} min | {reasons} |"
        )
    display(HTML("<pre>" + "\n".join(rows) + "</pre>"))
else:
    print(f"Error: {resp.text[:500]}")

In [ ]:
# Display Results: Gate Score Comparison

if not recommendations:
    print("No recommendations available.")
else:
    gate_ids = [r["gate_id"] for r in recommendations]
    scores = [r["score"] for r in recommendations]
    taxi_times = [r["taxi_time"] for r in recommendations]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Bar chart: scores ---
    colors = ["#2ecc71" if s == max(scores) else "#3498db" for s in scores]
    bars = axes[0].barh(gate_ids[::-1], scores[::-1], color=colors[::-1], edgecolor="white")
    axes[0].set_xlabel("Score")
    axes[0].set_xlim(0, 1.0)
    axes[0].set_title(f"Gate Recommendation Scores (flight: {SELECTED_ICAO24})")
    for bar, score in zip(bars, scores[::-1]):
        axes[0].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f"{score:.2f}", va="center", fontsize=10)

    # --- Bar chart: taxi times ---
    axes[1].barh(gate_ids[::-1], taxi_times[::-1], color="#e67e22", edgecolor="white")
    axes[1].set_xlabel("Taxi Time (minutes)")
    axes[1].set_title("Estimated Taxi Time per Gate")
    for i, (gt, tt) in enumerate(zip(gate_ids[::-1], taxi_times[::-1])):
        axes[1].text(tt + 0.2, i, f"{tt} min", va="center", fontsize=10)

    plt.suptitle(f"Gate Recommendations for {SELECTED_ICAO24}", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Best recommendation
    best = recommendations[0]
    print(f"\nBest gate: {best['gate_id']} (score: {best['score']:.2f}, taxi: {best['taxi_time']} min)")
    print(f"Reasons: {', '.join(best['reasons'])}")